# RescueVision v2 - event split map extract from old tar

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import subprocess, io, time
import pandas as pd

CANDIDATES = ["/kaggle/input/notebooks/aliomarsaleh/rv2-xbd-native-preprocess/processed_xbd_native.tar"]
import glob as _glob
for _p in _glob.glob("/kaggle/input/**/processed_xbd_native.tar", recursive=True):
    if _p not in CANDIDATES:
        CANDIDATES.append(_p)
TAR = next((p for p in CANDIDATES if Path(p).exists()), CANDIDATES[0])
print("tar:", TAR)
mark("evmap:tar", extra=TAR)

# the ONLY read pattern that survives on this mount: plain full extract (proven
# by the joint/building trainers); python streaming/member-scan gets the session killed
t0 = time.time()
subprocess.run(["tar", "-xf", TAR, "-C", "/tmp"], check=True)
print("extracted in", round(time.time() - t0), "s")
mark("evmap:extracted", extra=f"{round(time.time()-t0)}s")

t = pd.read_csv("/tmp/processed_xbd/metadata/tiles.csv")
s = pd.read_csv("/tmp/processed_xbd/metadata/splits_event_aware.csv")
m = t.merge(s, on="tile_id")
ev = dict(zip(m["disaster_name"], m["split"]))
(WORK / "event_split.json").write_text(json.dumps(ev))
print(len(ev), "events; splits:", pd.Series(ev).value_counts().to_dict())
mark("evmap:done", extra=str(pd.Series(ev).value_counts().to_dict()))